<a href="https://oscar-defelice.github.io">
<img src="../../img/image.png" height="125" align="right" />
</a>

# TP 01 — Tokenisation, Text Normalisation & Datasets

**Course:** Natural Language Processing  
**Session:** 1 / 10  
**Duration:** ~2h (TP portion)

---

## Objectives

By the end of this session you will be able to:

- Apply a reproducible text normalisation pipeline
- Compare tokenisation strategies and measure their impact on vocabulary size and OOV rate
- Load a text classification dataset, inspect it critically, and build a proper train/val/test split
- Implement a majority-class baseline and interpret the resulting metrics
- Write a short error analysis

---

## Roadmap

| Part | Task | Deliverable |
|------|------|-------------|
| 1 | Text normalisation | Implement `preprocess_text` + pass assertions |
| 2 | Tokenisation comparison | Stats table: vocab size, mean length, OOV rate |
| 3 | Dataset inspection & split | Class distribution plot + justified split |
| 4 | Majority-class baseline | Classification report + confusion matrix |
| 5 | Error analysis | Written analysis of 10 misclassified examples |

Each part ends with a **📝 Your analysis** cell. Fill these in — they are reviewed at the end of the session.

---

## Ground rules

- Every function must have its docstring filled in **before** you write the body.
- Do not look at the test set before Part 4.
- When you hit an error, read it carefully before asking for help.

---

## Setup

In [ ]:
# Uncomment if running on Colab
# !pip install datasets -q

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    accuracy_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Dataset ───────────────────────────────────────────────────────────────────
# Your instructor will announce the dataset at the start of class.
# Supported values:
#   "ag_news"   → 4-class news topic classification (recommended for this session)
#   "imdb"      → binary sentiment
#   "emotion"   → 6-class emotion in tweets  (dair-ai/emotion)
#
DATASET_NAME = "ag_news"   # ← CHANGE THIS IF INSTRUCTED

print(f"Dataset  : {DATASET_NAME}")
print("Setup OK.")

---
## Part 1 — Text Normalisation

Before any model sees text, it goes through a normalisation pipeline. Small decisions here (lowercase or not? keep punctuation?) have downstream effects on vocabulary size and model performance.

We will build **one reusable function** and use it throughout the course.

### 1.1 — Implement `preprocess_text`

In [ ]:
def preprocess_text(text: str) -> str:
    """
    Apply a minimal, reproducible normalisation pipeline to a single string.

    Steps (apply in this exact order):
      1. Lowercase all characters
      2. Remove HTML tags (e.g. <br />, <p>, </div>)
      3. Replace any character that is not a letter, digit, or space
         with a single space
      4. Collapse multiple consecutive whitespace characters into one space
      5. Strip leading and trailing whitespace

    Parameters
    ----------
    text : str
        Raw input string.

    Returns
    -------
    str
        Normalised string.

    Examples
    --------
    >>> preprocess_text("Hello,   World! <br/>")
    'hello world'
    >>> preprocess_text("  NLP is FUN!!!  ")
    'nlp is fun'
    >>> preprocess_text("<p>U.S. stocks rose 2.3%</p>")
    'u s stocks rose 2 3'
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# ── Assertions — all must pass before you continue ────────────────────────────
assert preprocess_text("Hello,   World! <br/>")       == "hello world",       "Failed: HTML + punctuation"
assert preprocess_text("  NLP is FUN!!!  ")            == "nlp is fun",        "Failed: leading/trailing spaces + punctuation"
assert preprocess_text("<p>U.S. stocks rose 2.3%</p>") == "u s stocks rose 2 3", "Failed: HTML + mixed punctuation"
assert preprocess_text("already clean")               == "already clean",     "Failed: clean input should be unchanged"
assert preprocess_text("")                             == "",                  "Failed: empty string"
print("All assertions passed ✓")

### 1.2 — Ablation: what happens if we skip a step?

Understanding *why* each step matters is as important as implementing it.

In [ ]:
def preprocess_no_lowercase(text: str) -> str:
    """
    Same pipeline as preprocess_text but WITHOUT the lowercase step.

    Used to measure the effect of case folding on vocabulary size.
    """
    # YOUR CODE HERE  (copy-paste and remove step 1)
    raise NotImplementedError


def preprocess_keep_punctuation(text: str) -> str:
    """
    Same pipeline as preprocess_text but keeping punctuation characters.

    Only apply steps 1, 2, 4, and 5 (skip the punctuation removal step).
    """
    # YOUR CODE HERE
    raise NotImplementedError

### 📝 Your analysis — Part 1

**Which step in the pipeline has the biggest impact on vocabulary size, and why?**

> *[Your answer here]*

**Give an example where removing punctuation loses information** that a classifier might need.

> *[Your answer here]*

---
## Part 2 — Tokenisation Comparison

Tokenisation is the first design decision in any NLP pipeline. We will compare three strategies on the same corpus and measure their properties quantitatively.

The three tokenisers:

| Name | Strategy | Library |
|------|----------|---------|
| `whitespace` | Split on spaces | pure Python |
| `nltk` | Rule-based word tokeniser | `nltk` |
| `bpe` | GPT-4 byte-pair encoding | `tiktoken` |

In [ ]:
# Uncomment to install if needed
# !pip install nltk tiktoken -q

import nltk
import tiktoken
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Initialise the BPE tokeniser once (it's slow to load)
BPE_ENC = tiktoken.encoding_for_model("gpt-4")

### 2.1 — Implement tokeniser wrappers

In [ ]:
def tokenize_whitespace(text: str) -> list[str]:
    """
    Tokenise by splitting on whitespace.

    Apply preprocess_text first, then split on spaces.
    Returns a list of string tokens.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def tokenize_nltk(text: str) -> list[str]:
    """
    Tokenise using NLTK's word_tokenize.

    Apply preprocess_text first, then use nltk.word_tokenize.
    Returns a list of string tokens.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def tokenize_bpe(text: str) -> list[int]:
    """
    Tokenise using GPT-4's BPE encoding (tiktoken).

    Do NOT apply preprocess_text first — BPE operates on raw text.
    Returns a list of integer token IDs.

    Hint: use BPE_ENC.encode(text)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Quick sanity check
sample = "The Federal Reserve raised interest rates by 0.25%."
print("whitespace :", tokenize_whitespace(sample))
print("nltk       :", tokenize_nltk(sample))
print("bpe        :", tokenize_bpe(sample))

### 2.2 — Build the comparison table

We compare the tokenisers on a sample of the corpus. Computing over the full dataset would take too long for BPE.

In [ ]:
def compare_tokenisers(
    texts: list[str],
    tokenisers: dict,
) -> pd.DataFrame:
    """
    Compute statistics for each tokeniser on a list of texts.

    For each tokeniser, compute:
      - vocab_size    : number of unique tokens across all texts
      - mean_length   : mean number of tokens per text
      - median_length : median number of tokens per text
      - oov_proxy     : fraction of unique tokens that appear only once
                        (hapax legomena — a proxy for OOV rate on unseen data)

    Parameters
    ----------
    texts      : list of raw strings
    tokenisers : dict mapping name (str) → callable that takes str and returns list

    Returns
    -------
    pd.DataFrame with one row per tokeniser and the four columns above.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Load a small sample to keep runtime under 60 seconds
# (we'll load the full dataset in Part 3)
sample_raw = load_dataset(DATASET_NAME, split="train[:500]")
sample_texts = sample_raw["text"]

tokenisers = {
    "whitespace": tokenize_whitespace,
    "nltk"      : tokenize_nltk,
    "bpe"       : tokenize_bpe,
}

comparison = compare_tokenisers(sample_texts, tokenisers)
comparison

### 2.3 — Vocabulary growth curve

How quickly does each tokeniser's vocabulary saturate as we see more text?

In [ ]:
def plot_vocab_growth(
    texts: list[str],
    tokenisers: dict,
    step: int = 20,
) -> None:
    """
    Plot vocabulary size as a function of number of texts seen.

    For each tokeniser, iterate over texts in chunks of `step`,
    and after each chunk record the cumulative vocabulary size.
    Plot one line per tokeniser.

    Parameters
    ----------
    texts      : list of raw strings
    tokenisers : dict mapping name → callable
    step       : record vocabulary size every `step` texts
    """
    # YOUR CODE HERE
    raise NotImplementedError


plot_vocab_growth(sample_texts, tokenisers, step=25)

### 📝 Your analysis — Part 2

**Compare the three tokenisers on vocabulary size and mean length.** Which produces the largest vocabulary? Which produces the shortest sequences? Is there a trade-off?

> *[Your answer here]*

**Why does BPE operate on raw text rather than preprocessed text?** What would happen if you applied `preprocess_text` before BPE?

> *[Your answer here]*

**Which tokeniser would you choose** for (a) a classical ML model like logistic regression, and (b) a Transformer? Justify briefly.

> *[Your answer here]*

---
## Part 3 — Dataset Inspection & Split

A model is only as good as the data it is trained on. Before fitting anything, we need to understand what we are working with.

### 3.1 — Load the full dataset

In [ ]:
def load_text_dataset(name: str) -> tuple[pd.DataFrame, dict]:
    """
    Load a HuggingFace text classification dataset into a unified DataFrame.

    Returns
    -------
    df : pd.DataFrame
        Columns: 'text' (str), 'label' (int), 'label_name' (str), 'split' (str)
    meta : dict
        Keys: 'label_names' (list[str]), 'num_classes' (int), 'total_examples' (int)
    """
    configs = {
        "ag_news" : {"label_names": ["World", "Sports", "Business", "Sci/Tech"], "text_col": "text"},
        "imdb"    : {"label_names": ["Negative", "Positive"],                   "text_col": "text"},
        "emotion" : {"label_names": ["sadness", "joy", "love", "anger", "fear", "surprise"], "text_col": "text"},
    }
    if name not in configs:
        raise ValueError(f"Unknown dataset '{name}'. Choose from: {list(configs)}")

    cfg = configs[name]
    raw = load_dataset(name if name != "emotion" else "dair-ai/emotion")

    frames = []
    for split_name, split_data in raw.items():
        df_split = split_data.to_pandas().rename(columns={cfg["text_col"]: "text"})
        df_split["split"]      = split_name
        df_split["label_name"] = df_split["label"].apply(lambda i: cfg["label_names"][i])
        frames.append(df_split[["text", "label", "label_name", "split"]])

    df   = pd.concat(frames, ignore_index=True)
    meta = {"label_names": cfg["label_names"], "num_classes": len(cfg["label_names"]), "total_examples": len(df)}
    return df, meta


df, meta = load_text_dataset(DATASET_NAME)
print(f"Total   : {meta['total_examples']:,}")
print(f"Classes : {meta['label_names']}")
print(f"Splits  : {df['split'].value_counts().to_dict()}")
df.head(3)

### 3.2 — Compute per-class statistics

In [ ]:
def compute_dataset_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-class statistics on the training split.

    For each class compute:
      count, proportion (0–1), mean_length, median_length, min_length, max_length
    where 'length' = number of whitespace-split tokens in the raw text.

    Parameters
    ----------
    df : full dataset DataFrame (all splits)

    Returns
    -------
    pd.DataFrame — one row per class
    """
    # YOUR CODE HERE
    raise NotImplementedError


stats = compute_dataset_stats(df)
stats

### 3.3 — Visualise class distribution and text lengths

In [ ]:
def plot_class_distribution(df: pd.DataFrame, split: str = "train") -> None:
    """
    Bar chart of class counts for a given split.

    Requirements:
      - Bars sorted by frequency (descending)
      - Count label on top of each bar
      - Title includes split name and total count
    """
    # YOUR CODE HERE
    raise NotImplementedError


def plot_length_distribution(df: pd.DataFrame, split: str = "train", max_tokens: int = 200) -> None:
    """
    Overlapping histograms of whitespace-token counts, one per class.

    Requirements:
      - Clip lengths at max_tokens for readability
      - Semi-transparent bars (alpha=0.5)
      - Legend, axis labels, title
    """
    # YOUR CODE HERE
    raise NotImplementedError


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plt.sca(axes[0]); plot_class_distribution(df, split="train")
plt.sca(axes[1]); plot_length_distribution(df, split="train")
plt.tight_layout()

### 3.4 — Train / validation / test split

In [ ]:
def make_splits(
    df: pd.DataFrame,
    val_size: float = 0.1,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build train / validation / test DataFrames.

    Rules:
      - Reuse existing 'test' split if present; do not touch it.
      - Reuse existing 'validation' split if present.
      - If no 'validation' exists, carve one out of 'train'
        using stratified sampling (val_size fraction).

    Returns
    -------
    train_df, val_df, test_df
    """
    # YOUR CODE HERE
    raise NotImplementedError


train_df, val_df, test_df = make_splits(df, val_size=0.1)

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Test  : {len(test_df):,}")

# Verify stratification
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    props = split["label_name"].value_counts(normalize=True).round(3).to_dict()
    print(f"{name}: {props}")

### 📝 Your analysis — Part 3

**Describe the dataset in 3 sentences.** Size, task, class balance, typical text length.

> *[Your answer here]*

**Flag one potential issue** in the data that could affect training or evaluation.

> *[Your answer here]*

**Why must the test split be stratified?** What would happen to the reported accuracy if one class were over-represented in the test set?

> *[Your answer here]*

---
## Part 4 — Majority-Class Baseline

> *"Always compare against the stupidest possible model first."*

A majority-class baseline predicts the most frequent training label for every input, ignoring the text entirely. Any model that cannot beat this baseline does not have a learning problem — it has a data problem.

### 4.1 — Implement and evaluate

In [ ]:
def majority_baseline(
    train_df: pd.DataFrame,
    eval_df: pd.DataFrame,
) -> tuple[list, list]:
    """
    Predict the most frequent class in train_df for every example in eval_df.

    Returns
    -------
    y_true : list[int] — ground truth labels from eval_df
    y_pred : list[int] — predicted labels (constant, equal to majority class)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Evaluate on validation set first
y_true_val, y_pred_val = majority_baseline(train_df, val_df)
print("=== Validation ===")
print(classification_report(y_true_val, y_pred_val, target_names=meta["label_names"], zero_division=0))

In [ ]:
# ── Test set — run once, do not use results to tune anything ──────────────────
y_true_test, y_pred_test = majority_baseline(train_df, test_df)
print("=== Test ===")
print(classification_report(y_true_test, y_pred_test, target_names=meta["label_names"], zero_division=0))

### 4.2 — Confusion matrix

In [ ]:
def plot_confusion_matrix(
    y_true: list,
    y_pred: list,
    label_names: list[str],
    title: str = "Confusion Matrix",
    normalise: bool = True,
) -> None:
    """
    Plot a confusion matrix.

    When normalise=True, rows are normalised to sum to 1
    (i.e. each cell shows recall for that class).
    Display values with 2 decimal places when normalised, integers otherwise.
    Use a diverging colourmap (e.g. 'Blues').
    """
    # YOUR CODE HERE
    raise NotImplementedError


plot_confusion_matrix(y_true_test, y_pred_test, meta["label_names"], title="Majority Baseline — Test")

### 📝 Your analysis — Part 4

**Interpret macro-F1 vs weighted-F1.** They are different numbers here. Why? Which should you report?

> *[Your answer here]*

**Theoretical ceiling.** On a perfectly balanced K-class dataset, what is the maximum accuracy a majority classifier can achieve? How does your dataset compare to this ceiling?

> *[Your answer here]*

---
## Part 5 — Error Analysis

Error analysis is how you turn metric numbers into insight. Even on a trivial baseline, it tells you which examples the model systematically ignores — and why.

### 5.1 — Sample misclassified examples

In [ ]:
def find_errors(
    eval_df: pd.DataFrame,
    y_true: list,
    y_pred: list,
    label_names: list[str],
    n: int = 10,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Return a stratified sample of misclassified examples.

    'Stratified' means the n examples cover multiple true classes —
    not all from the same class.

    Returns a DataFrame with columns:
      'text'       : raw text
      'true_label' : ground truth class name
      'predicted'  : predicted class name
      'num_tokens' : whitespace token count
    """
    # YOUR CODE HERE
    raise NotImplementedError


errors = find_errors(test_df, y_true_test, y_pred_test, meta["label_names"], n=10)
pd.set_option("display.max_colwidth", 120)
errors

### 5.2 — Vocabulary overlap between classes

In [ ]:
def compute_vocab_overlap(train_df: pd.DataFrame, top_n: int = 500) -> pd.DataFrame:
    """
    Pairwise Jaccard similarity of per-class vocabularies.

    For each class, build a vocabulary from its top_n most frequent
    whitespace-split tokens. Then compute:

        J(A, B) = |vocab_A ∩ vocab_B| / |vocab_A ∪ vocab_B|

    Returns a symmetric DataFrame (rows = cols = class names).
    """
    # YOUR CODE HERE
    raise NotImplementedError


overlap = compute_vocab_overlap(train_df, top_n=500)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(overlap, annot=True, fmt=".2f", cmap="Blues", ax=ax, vmin=0, vmax=1)
ax.set_title("Vocabulary Jaccard Overlap — top-500 tokens per class")
plt.tight_layout()
plt.show()

### 📝 Your analysis — Part 5

**Describe a pattern in the 10 errors.** Do they share surface characteristics (length, topic, vocabulary)? Why does the majority baseline fail on them specifically?

> *[Your answer here]*

**What does the vocabulary overlap tell you?** Which pair of classes overlaps most? Does this match your intuition about the task?

> *[Your answer here]*

**One thing that surprised you** about this dataset that you did not expect at the start.

> *[Your answer here]*

---
## Summary

Run this cell to auto-generate your results table before the end-of-session review.

In [ ]:
summary = pd.DataFrame({
    "Metric": ["Accuracy", "Macro F1", "Weighted F1"],
    "Majority Baseline (test)": [
        f"{accuracy_score(y_true_test, y_pred_test):.3f}",
        f"{f1_score(y_true_test, y_pred_test, average='macro',    zero_division=0):.3f}",
        f"{f1_score(y_true_test, y_pred_test, average='weighted', zero_division=0):.3f}",
    ],
})

print(f"Dataset    : {DATASET_NAME}")
print(f"Train size : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}")
print(f"Classes    : {meta['num_classes']}  ({', '.join(meta['label_names'])})")
print()
display(summary)
print()
print("Tokeniser comparison:")
display(comparison)

---
## What's next?

In **TP 02** you will train n-gram language models on this dataset and measure perplexity.
In **TP 03** you will replace the majority baseline with a Naive Bayes classifier.

**Keep this notebook.** The `train_df`, `val_df`, `test_df`, `meta`, and `comparison` objects carry forward — later sessions will import and reuse them.